# Beyond the Stopwatch (V2 Champion): Colab GPU 1D-CNN + Attention & 10 Hz Optical Kinematics
**NFL Big Data Bowl 2027 ($100,000 Prize Pool) | Reproducible Official-Data & Deep Temporal Fusion Notebook**

This notebook executes our complete, strictly zero-leakage V2 pipeline directly on the official NFL Big Data Bowl 2027 dataset (~2.2 GiB across 9 CSVs: 510 rookie prospects, 431,094 optical Combine player tracking frames, 314,197 NFL play-level outcomes, and 22,654,290 regular-season NFL game tracking frames from 2023 to 2025), fusing our Colab NVIDIA L4 GPU PyTorch Multi-Scale 1D-CNN + Attention Sequence Encoder with Polars physics kinematics.

In [ ]:
import sys
from pathlib import Path

KAGGLE_DIR = Path("/kaggle/input/nfl-big-data-bowl-2027")
LOCAL_DIR = Path("/Volumes/Extra/datasets/kaggle-nfl2027/nfl-big-data-bowl-2027")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Using official data directory: {DATA_DIR}")

In [ ]:
from nfl_bdb2027.data import DatasetPaths, audit_dataset_inventory, load_combine_results
from nfl_bdb2027.kinematics import extract_rep_level_kinematics
from nfl_bdb2027.outcomes import build_outcomes_table
from nfl_bdb2027.models import run_full_modeling_suite
from nfl_bdb2027.replay import build_film_room_replay_bundle
from nfl_bdb2027.charts import generate_all_figures
from nfl_bdb2027.report import write_metrics_json, write_kaggle_writeup_md, write_interactive_html_report

paths = DatasetPaths.from_dir(DATA_DIR)
inventory = audit_dataset_inventory(paths)
print(f"Prospects: {inventory['total_prospects']} | Combine Player Frames: {inventory['combine_tracking_player_rows']:,}")
print(f"Play Rows: {inventory['player_play_rows']:,} | NFL Game Tracking Frames: {inventory['game_tracking_total_rows']:,}")
print(f"40-yd Opt-Outs: {inventory['forty_opt_out_count']} | 3-Cone Opt-Outs: {inventory['three_cone_opt_out_count']} | Skill-Drill Coverage: {inventory['combine_tracking_coverage']['players_with_skill_tracking']}/510")

In [ ]:
# Extract 10 Hz optical Combine kinematics & regular-season NFL outcomes (100% Polars lazy/vectorized)
rep_df = extract_rep_level_kinematics(paths)
cr_pl = load_combine_results(paths)
outcomes_df = build_outcomes_table(paths)
print(f"Drill reps extracted: {rep_df.height:,} | Outcomes table shape: {outcomes_df.shape}")

In [ ]:
# Run strictly zero-leakage Stratified 5-Fold & Temporal LOYO CV with Colab L4 GPU PyTorch 1D-CNN + Attention Fusion
# (To retrain the GPU encoder from scratch on 431,094 10 Hz frames, run scripts/train_temporal_cnn_colab.py --mode dispatch)
results, master_df = run_full_modeling_suite(outcomes_df, rep_df, cr_pl, n_boot=2000)

import pandas as pd
rows = []
for bkey, label in [
    ("benchmark_1_game_kinematic_translation", "B1: Sunday Game Kinematic Power"),
    ("benchmark_2_nfl_career_value", "B2: Exposure-Normalized NFL Career Value"),
]:
    for mkey, mv in results[bkey]["stratified_5fold"].items():
        tv = results[bkey]["temporal_loyo"][mkey]
        rows.append({
            "Benchmark": label,
            "Modality": mkey,
            "Strat R2": round(mv["r2"], 4),
            "Strat Rho": round(mv["spearman_rho"], 4),
            "Temporal R2": round(tv["r2"], 4),
            "Temporal Rho": round(tv["spearman_rho"], 4),
        })
pd.DataFrame(rows)

In [ ]:
# Inspect real named Day-3/UDFA Optical Gems, 40-Yard Opt-Out Stars & Film-Room 10 Hz Matchups
display_cols = ["display_name", "draft_year", "combine_position", "draft_round", "draft_overall_pick", "forty", "opt_power_index", "opt_cbi_index", "snaps_per_season", "draft_capital_alpha"]
print("=== Top Day-3 / UDFA Optical Gems ===")
print(pd.DataFrame(results["case_studies"]["day3_udfa_optical_gems"])[display_cols].head(6).to_string(index=False))
print("\n=== Top 40-Yard Dash Opt-Out Stars ===")
print(pd.DataFrame(results["case_studies"]["forty_opt_out_stars"])[display_cols].head(5).to_string(index=False))

In [ ]:
# Render all 5 Economist/NYT pure-white SVGs, metrics v2 JSON, Writeup & Canvas 2D Film-Room HTML Report
from IPython.display import SVG, display
out_dir = Path("../../materials").resolve() if Path("../../materials").exists() else Path("./outputs")
replay_scenarios = build_film_room_replay_bundle(paths, master_df)
fig_paths = generate_all_figures(paths, results, master_df, figures_dir=out_dir / "figures")
write_metrics_json(inventory, results, out_path=out_dir / "real_data_metrics_v2.json")
write_kaggle_writeup_md(inventory, results, master_df, out_path=out_dir / "NFL_BDB2027_Kaggle_Writeup.md")
write_interactive_html_report(inventory, results, master_df, fig_paths, replay_scenarios=replay_scenarios, out_path=out_dir / "NFL_BDB2027_Interactive_Report.html")

for k, fp in fig_paths.items():
    print(f"Rendered {k}: {fp}")
    display(SVG(filename=str(fp)))